### Overview of Transformer Engine, FP8, and Micro‑scaled FP8 (MXFP8)
---

<center><div style="text-align:center; color:#FF0000; border:3px solid red;height:50px;"> <b> Please run this Jupyter Notebook and the next lab by switching to the Transformer Engine Container</b> </div></center>

In this notebook, you will learn about the transformer engine, FP8 format, dynamic range, and mixed precision. The goal is to provide a solid foundation in these topics before proceeding with the optimization process.  

#### Background
[Transformer Engine (TE)](https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/index.html) is a library for accelerating Transformer models on NVIDIA GPUs, including support for 8-bit floating-point (FP8) precision on Hopper, Ada, and Blackwell GPUs, to deliver better performance with lower memory utilization in both training and inference. TE provides a collection of highly optimized building blocks for popular Transformer architectures and an automatic mixed-precision API that seamlessly integrates with your framework-specific code. TE also includes a framework-agnostic C++ API that can be integrated with other deep learning libraries to enable FP8 support for Transformers. With the Hopper GPU architecture, FP8 precision was introduced, offering improved performance over FP16 with no degradation in accuracy. TE provides a Python API consisting of modules for easily building a Transformer layer, as well as a framework-agnostic C++ library with structs and kernels for FP8 support. Modules provided by TE internally maintain the scaling factors and other values needed for FP8 training, greatly simplifying mixed-precision training for users.

#### Benefit of Transformer Engine
- Easy-to-use modules for building Transformer layers with FP8 support
- Optimizations (e.g. fused kernels) for Transformer models
- Support for FP8 on NVIDIA Hopper, Ada, and Blackwell GPUs
- Support for optimizations across all precisions (FP16, BF16) on NVIDIA Ampere GPU architecture generations and later

A sample PyTorch implementation is given below:

```python
import torch
import transformer_engine.pytorch as te
from transformer_engine.common import recipe

# Set dimensions.
in_features = 768
out_features = 3072
hidden_size = 2048

# Initialize model and inputs.
model = te.Linear(in_features, out_features, bias=True)
inp = torch.randn(hidden_size, in_features, device="cuda")

# Create an FP8 recipe. Note: All input args are optional.
fp8_recipe = recipe.DelayedScaling(margin=0, fp8_format=recipe.Format.E4M3)

# Enable autocasting for the forward pass
with te.autocast(enabled=True, recipe=fp8_recipe):
    out = model(inp)

loss = out.sum()
loss.backward()

```

In [ ]:
!python ../source_code/fp8/te_example.py

### Introduction to FP8 And Binary Interchange Format

FP8 is an 8-bit floating-point format designed for efficient memory usage and faster computation in deep learning, especially for AI models and hardware. Unlike standard floating-point formats such as FP16 or FP32, FP8 uses only 8 bits per number, resulting in significant resource savings but lower precision. 

#### FP8 Structure
- Sign bit: 1 bit (for positive/negative values)
- Exponent bits: Either 4 or 5 bits (controlling numeric range)
- Mantissa/Precision bits: Either 3 or 2 bits (controlling granularity of values)

The FP8 consists of two encodings/datatypes useful in different parts of neural networks:

- **E4M3** - it consists of 1 sign bit, 4 exponent bits, and 3 bits of mantissa. It can store values up to `+/-448` and nan.
- **E5M2** - it consists of 1 sign bit, 5 exponent bits, and 2 bits of mantissa. It can store values up to `+/-57344`, `+/- inf`, and `nan`. The trade-off for the increased dynamic range is lower precision in the stored values.
  
*Note: The "mantissa" denotes the IEEE 754 standard’s trailing significand field (i.e., bits not including the implied leading 1 bit for normal floating point numbers)*

During neural network training, both encoding methods may be utilized. Typically, forward activations and weights require more precision, so the `E4M3` datatype is best used during the forward pass. In the backward pass, however, gradients flowing through the network are typically less susceptible to precision loss but require a higher dynamic range. Therefore, they are best stored using the `E5M2` data format. H100 Tensor Cores support any combination of these types as inputs, enabling us to store each tensor in its preferred precision.

<center><img src="images/FP8-datatype.png" width="500" height="500" alt="FP8-datatypes"></center>
<center>Structure of the floating point datatypes</center><br/>

**Calculating the bit value of E4M3**
- Estimate the Exponent bias (for 4 bits): $2^{4-1}$ - 1 = 7
- Exponent decoded: 0101 (binary) = 5, exponent = 5 - 7 = -2
- Mantissa: convert 101 binary to fraction = 1 x $2^{-1}$ + 0 x $2^{-2}$ + 1 x $2^{-3}$ = 0.625
- The bits Value is: $(-1)^{0}$ x (1 + 0.625) x $2^{-2}$ = **0.40625**

**Calculating the bit value of E5M2**
- Estimate the Exponent bias (for 5 bits): $2^{5-1}$ - 1 = 15
- Exponent decoded: 01101 (binary) = 13, exponent = 13 - 15 = -2
- Mantissa: convert 10 binary to fraction = 1 x $2^{-1}$ + 0 x $2^{-2}$ = 0.5
- The bits Value is: $(-1)^{0}$ x (1 + 0.5) x $2^{-2}$ = **0.375**

FP8's use of exponents and mantissas makes it well-suited for representing values over a wide range. This is extremely important for neural network quantization, where preserving the model’s numeric range while losing minimal precision is crucial for accuracy.



#### FP8 Value Representation

Let's consider steps for storing a decimal number, for example, `5.5` in E4M3 format.

- Convert the decimal number `(5.5)` to binary. The fraction `.5` in binary is `.1 (because 1/2 = 0.5).` So, `5.5` in binary is `101.1`. 
- Next, move the decimal point to the left until only one digit (1) is in front of it. Hence, `101.1` becomes 1.011 x $2^{2}$. Since the decimal was moved 2 places to the left, this resulted in $2^{2}$. The part after the decimal point is considered the **mantissa (or significand)**: `011`.
- The number `5.5` is positive, so the **sign bit is `0`**. The exponent is `2` from the last step and is stored with a bias to allow for both positive and negative exponents. For E4M3, the bias is typically `7` (calculated as $2^{4-1}$ - 1). We store the actual exponent plus the bias. So, 2 + 7 = 9. The number 9 in 4-bit binary is **`1001` and represents the exponent field**. **The mantissa representation is `011`**. There exist exactly 3 bits, so it can be stored directly (if it were longer, it would be rounded up).
- Assemble the binary sequence for Sign bit, Exponent bits, and Mantissa bits. The number `5.5` is represented in the `E4M3` format as a binary sequence, `01001011`.  
  |Sign bit |Exponent bits   | Mantissa bits  |
  |  --  |  --  |  --  |
  |0  | 1001 | 011|

Let's illustrate the steps for storing a decimal number, such as `5.5`, in `E5M2 format` and how it differs from E4M3.

- Convert 5.5 to binary as 101.1.
- Write the binary scientific notation by moving the decimal point two places to get 1.011 x $2^{2}$.
- Get all the bits:
    - 5.5 is a positive number, so the **sign bit is `0`**
    - The exponent is 2 and the bias for E5M2 is `15` (calculated as $2^{5-1}$ - 1 = 16 - 1 = 15). Hence, the exponent plus the bias represents the **`actual exponent as: 2 + 15 = 17`**. The number 17 in 5-bit binary is `10001`.
    - Here is where the difference appears. The mantissa from the scientific notation is 011. But we only have 2 bits to store it in E5M2. So the leftover will be truncated or rounded. Therefore, only the first two bits: **`01`, are retained as the mantissa bit** while the trailing 1 is lost.
- Assemble the binary sequence for Sign bit, Exponent bits, and Mantissa bits. The number `5.5` is represented in the `E5M2` format as a binary sequence, `01000101`.  
  |Sign bit |Exponent bits   | Mantissa bits  |
  |  --  |  --  |  --  |
  |0  | 10001 | 01|

##### Loss of Precision
Because we lost that last `1` in the mantissa, the representation isn't perfect. We stored `1.01` in the mantissa instead of `1.011`. `1.01` in binary is `1.25` in decimal; therefore, the number we actually stored is 1.25 x $2^{2}$ = 1.25 x 4 = 5.0. In E5M2, the number `5.5` cannot be represented exactly. It would have to be rounded to `5.0`. This shows the trade-off. E5M2 can store massive numbers, but with less accuracy than E4M3.

#### Dynamic Range

Dynamic range in FP8 denotes the range from the smallest positive normal value to the largest finite value  the format can represent before overflow occurs. For E4M3 definition: The smallest positive normal value is $2^{-6}$ = `0.015625`, while the largest finite value is about `448`. In the case of E5M2 with more exponent bits and fewer mantissa bits, it can reach much larger values. The smallest positive normal value is ≈ $2^{-14}$ approximately `0.000061` while the largest finite value is `≈ 57,344`.

Let's estimate the largest finite value for E4M3:

- **Format Structure**: E4M3 uses 1 sign bit, 4 exponent bits, and 3 mantissa bits.
- **Bit Pattern**: The maximum finite value is represented by the bit pattern 0 1111 110.
    - Sign: 0; exponent bits: `1111`; and Mantissa bits: `111`, because one bit will be used for storing the `nan`, then the maximum mantissa bits will be represented as `110`.
    - Estimate the Exponent bias (for 4 bits): $2^{4-1}$ - 1 = 7
    - Exponent decoded: 1111 (binary) = 15, exponent = 15 - 7 = 8
    - Mantissa: convert 110 binary to fraction = 1 x $2^{-1}$ + 1 x $2^{-2}$ + 0 x $2^{-3}$ = 0.75
    - The Largest finite value is: $(-1)^{0}$ x (1 + 0.75) x $2^{8}$ = **448**

The largest finite value for E5M2 can also be calculated following the same step, the Sign be 0; Mantissa bits: `11`, and exponent bits: `11111`; because one bit will be used for storing the `Inf/NaN`, then the maximum exponent bits will be represented as `11110`. The Largest finite value is: $(-1)^{0}$ x (1 + 0.75) x $2^{15}$ = **57,344**


### Mixed Precisions

In standard training, you do everything in FP32: activations, weights, gradients, optimizer state, and accumulations. Mixed precision changes this by: -1) Using a low‑precision format (traditionally FP16/BF16) for: Forward activations, Most weights (or a working copy of them), Many backward gradients, 2) Keeping higher‑precision (FP16/FP32) for: A master copy of weights, Accumulation in matrix multiplications and convolutions
Optimizer state (e.g., Adam moments), and Loss with some numerically sensitive operations. In FP8 mixed precision, you push even more of the work into 8‑bit floats while still relying on 16‑ or 32‑bit for stability‑critical parts like accumulations and master weights. The mixed-precision recipe for FP16 training has 2 components: choosing which operations to perform in FP16 and dynamic loss scaling. Dynamic loss scaling helps avoid both overflow and underflow in gradients during training. Those may happen because, while the dynamic range of FP16 is sufficient to store the distribution of gradient values, this distribution may be centered on values that are too high or too low for FP16 to handle. Scaling the loss shifts those distributions (without affecting numeric, using only powers of 2) into the FP16 range.

#### FP8 Mixed Precision Training

While the dynamic range provided by the FP8 types is sufficient to store any particular activation or gradient, it is not sufficient for all of them at the same time. This makes the single-loss scaling-factor strategy, which worked for FP16, infeasible for FP8 training and instead requires distinct scaling factors for each FP8 tensor. There are multiple strategies for choosing a scaling factor that is appropriate for a given FP8 tensor:

- **just-in-time scaling**: This strategy selects the scaling factor based on the maximum absolute value (amax) of the tensor being produced. In practice, it is infeasible because it requires multiple passes over the data: the operator produces and writes the output in higher precision, then the maximum absolute value of the output is found and applied to all values to obtain the final FP8 output. This results in significant overhead, severely diminishing the gains from using FP8.
- **delayed scaling**: This strategy chooses the scaling factor based on the maximums of absolute values seen in some number of previous iterations. This enables full performance of FP8 computation, but requires storing the history of maximums as additional parameters of the FP8 operators.

<center><img src="images/Delayed-scaling.png" width="500" height="500"  /></center>
<center>Delayed scaling strategy</center>


#### MXFP8 And Block Scaling

The NVIDIA Blackwell architecture introduced support for a new FP8 variant, [Micro‑scaled FP8 (MXFP8)](https://www.opencompute.org/documents/ocp-microscaling-formats-mx-v1-0-spec-final-pdf). MXFP8 is an 8‑bit floating‑point format with “micro‑scaling”: it stores numbers in FP8 but adds a shared scale factor per small block of values to keep accuracy while cutting memory and compute cost for AI models.

**FP8 vs Micro‑scaled FP8**

The main difference between “regular” FP8 and MXFP8 lies in the scaling granularity. In FP8, each tensor has a single FP32 scaling factor, so all values in the tensor need to “fit” within the dynamic range of the FP8 datatype. This requires using the less-precise E5M2 format to represent some tensors in the network (e.g., gradients). MXFP8 addresses this by assigning a different scaling factor to each block of 32 consecutive values. This allows all values to be represented with the E4M3 datatype.

<center><img src="images/MXFP8.png" width="600" height="600"  /></center>
<center>MXFP8 uses multiple scaling factors for a single tensor. The picture shows only 4 values per block for simplicity, but the real MXFP8 has 32 values per block.</center> <br/>


Another difference is the datatype used to store the scaling factors. FP8 uses FP32 (E8M23) while MXFP8 uses an 8-bit representation of a power of 2 (E8M0). You can read more about MXFP8 from the paper: [Recipes for Pre-training LLMs with MXFP8](https://arxiv.org/pdf/2506.08027v1)

<center><img src="images/E8M0.png" width="700" height="900"  /></center>
<center>Structure of the E8M0 datatype used for storing scaling factors in MXFP8.</center>



#### Handling Transposes

The forward and backward passes of linear layers involve multiple matrix multiplications with different reduction dimensions. Blackwell Tensor Cores require MXFP8 data to be “consecutive” over the reduction dimension, so MXFP8 training uses non-transposed and transposed MXFP8 tensors at different points. However, while transposing FP8 data is numerically trivial, transposing MXFP8 data requires `requantization`. To avoid loss of precision connected with this double quantization, Transformer Engine creates both regular and transposed copies of the tensor from the original high-precision input.

<center><img src="images/Transpose.png" width="600" height="600"  /></center>
<center>Linear layer in MXFP8. Calculating both forward and backward pass requires tensors quantized in both directions</center>

Now that we have a background in the Transformer Engine and FP8, let's proceed to learn how to apply these concepts to optimize our code. Please click on the `Next Notebook` link to proceed.


## <center><div style="text-align:center; color:#FF0000; border:3px solid red;height:80px;"> <b><br/> [Next Notebook](nsys-fp8.ipynb) </b> </div></center>

## References

- https://www.baseten.co/blog/fp8-efficient-model-inference-with-8-bit-floating-point-numbers/
- https://developer.nvidia.com/blog/floating-point-8-an-introduction-to-efficient-lower-precision-ai-training
- https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/examples/fp8_primer.html

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.